# Deep Research Tool — 典型的な使用例集

現場でよくある構成を、そのまま使えるコードで7例まとめました。
APIキー・パス類は自分の環境に合わせて書き換えてください。

| 例 | シナリオ |
|---|---|
| 1 | 最小構成のクイックリサーチ |
| 2 | 日本語ビジネスレポート（Word・15ページ・図表つき） |
| 3 | 社内プロキシ環境（APIゲートウェイ + Edge + ローカルWebDriver） |
| 4 | 手元の資料だけで調査（ローカル文書 / ハイブリッド） |
| 5 | 工程別LLMでコスト最適化 |
| 6 | フェルミ推定つき定量調査 |
| 7 | ローカルLLM（Ollama）で完全ローカル運用 |


## 例1: 最小構成のクイックリサーチ

DuckDuckGo検索 + Markdown出力。まず動かして全体の流れを掴む構成です。

In [ ]:
from deep_research_tool.config import create_config
from deep_research_tool.main import DeepResearchTool

config = create_config(
    provider="openai",
    openai_api_key="sk-...",
    search_region="jp-jp",
    research_iterations=2,      # 少なめにして高速に
    max_pages_per_query=2,
    output_format="markdown",
    output_dir="./output/quick",
)

result = DeepResearchTool(config).run(query="ペロブスカイト太陽電池の実用化動向")
print("→", result["report_path"])

## 例2: 日本語ビジネスレポート（Word・15ページ・図表つき）

V2生成エンジンで用語統一と文体の一貫性を確保し、図表を自動挿入した
docxを出力します。役員報告・部内共有向けの構成です。


In [ ]:
config = create_config(
    provider="openai",
    openai_api_key="sk-...",
    model="gpt-5-mini",
    search_region="jp-jp",
    research_iterations=3,
    # レポート
    report_generator_version="v2",
    output_format="docx",
    target_pages=15,
    v2_writing_style="business",     # です・ます調
    v2_target_audience="business",
    v2_enable_polish=True,           # 日本語推敲パス
    # 図表
    auto_figures=True,
    chart_library="seaborn",
    intelligent_charts=True,
    # 品質
    enable_verification=True,        # ハルシネーション検証
    evidence_format="both",
    output_dir="./output/biz_report",
)

def on_progress(msg, pct):
    print(f"[{pct:5.1f}%] {msg}")

result = DeepResearchTool(config).run(
    query="グリーン鉄鋼（水素還元製鉄）の世界動向と事業機会",
    requirements=(
        "主要各社の技術方式と量産時期、コスト構造、政策支援、"
        "2030年までの需要予測を含めること"
    ),
    progress_callback=on_progress,
)
print("→", result["report_path"])

## 例3: 社内プロキシ環境（APIゲートウェイ + Edge + ローカルWebDriver）

外部接続がプロキシ経由に制限され、LLM APIも社内ゲートウェイを通す環境の構成です。

ポイント:
1. `openai_base_url` — LLMリクエストの宛先を社内ゲートウェイに変更
2. `https_proxy` / `http_proxy` — Web検索・ページ取得のプロキシ
3. `verify_ssl=False` — SSLインスペクション（自己署名証明書）対策
4. `browser="edge"` + `driver_path` — 社内標準ブラウザ + 手動配置したWebDriver
   （プロキシ下では webdriver-manager の自動ダウンロードが失敗するため）


In [ ]:
config = create_config(
    provider="openai",
    openai_api_key="sk-...",
    openai_base_url="https://ai-gateway.example.co.jp/v1",  # ★社内ゲートウェイ
    # プロキシ
    http_proxy="http://proxy.example.co.jp:8080",
    https_proxy="http://proxy.example.co.jp:8080",
    verify_ssl=False,               # SSLインスペクション環境なら False
    # ブラウザ検索（JS描画サイトも読む）
    search_method="selenium",
    browser="edge",
    driver_path=r"C:\tools\msedgedriver.exe",   # ★手動配置したドライバ
    crawl_mode="ai_crawl_selenium",               # AIクロールも同じブラウザ設定を使用
    output_format="docx",
    output_dir="./output/intranet",
)

result = DeepResearchTool(config).run(query="競合他社の設備投資動向 2025-2026")
print("→", result["report_path"])

### WebDriverの入手先（手動配置）

| ブラウザ | ドライバ | 入手先 |
|---|---|---|
| Edge | msedgedriver | https://developer.microsoft.com/microsoft-edge/tools/webdriver/ |
| Chrome | chromedriver | https://googlechromelabs.github.io/chrome-for-testing/ |
| Firefox | geckodriver | https://github.com/mozilla/geckodriver/releases |

ブラウザ本体とドライバのバージョンを揃えてください（Edgeなら `edge://settings/help` で確認）。


## 例4: 手元の資料だけで調査（ローカル文書 / ハイブリッド）

`source_mode="local"` はWebに一切アクセスせず、渡したPDF・Excel等のみから
レポートを作ります。`"hybrid"` にするとWeb検索で補完します。


In [ ]:
config = create_config(
    provider="openai",
    openai_api_key="sk-...",
    source_mode="local",            # Webに出ない。"hybrid" でWeb併用
    report_generator_version="v2",
    output_format="docx",
    output_dir="./output/local_docs",
)

result = DeepResearchTool(config).run(
    query="添付の決算資料に基づく事業セグメント別の収益性分析",
    additional_documents=[
        "./docs/kessan_2025Q4.pdf",
        "./docs/segment_data.xlsx",
        "./docs/chuki_keikaku.pptx",
    ],
)
print("→", result["report_path"])

## 例5: 工程別LLMでコスト最適化

計画・評価は軽量モデル、執筆のみ高性能モデルにして、品質を保ちながら
APIコストを下げる構成です。


In [ ]:
config = create_config(
    provider="openai",
    openai_api_key="sk-...",
    model="gpt-5-mini",
    stage_llm={
        "planning":   {"provider": "openai", "model": "gpt-5-nano"},
        "crawling":   {"provider": "openai", "model": "gpt-5-nano"},
        "evaluation": {"provider": "openai", "model": "gpt-5-nano"},
        "writing":    {"provider": "openai", "model": "gpt-5"},
    },
    report_generator_version="v2",
    output_dir="./output/cost_optimized",
)

result = DeepResearchTool(config).run(query="全固体電池のサプライチェーン分析")
print("→", result["report_path"])

## 例6: フェルミ推定つき定量調査

「統計が存在しない数字」を要素分解して推定し、根拠と感度分析つきで
レポートに載せます。


In [ ]:
config = create_config(
    provider="openai",
    openai_api_key="sk-...",
    fermi_estimation=True,
    fermi_auto_detect=True,
    fermi_monte_carlo=1000,
    fermi_include_sensitivity=True,
    fermi_enable_sub_decomposition=True,
    report_generator_version="v2",
    output_format="docx",
    output_dir="./output/fermi",
)

result = DeepResearchTool(config).run(
    query="国内の産業用ドローン点検サービスの市場規模（2030年）",
    requirements="市場規模は金額ベースで推定し、推定根拠と感度分析を明示すること",
)
print("→", result["report_path"])

## 例7: ローカルLLM（Ollama）で完全ローカル運用

機密性の高い調査で外部APIを使えない場合、OllamaやvLLMで動かせます。
`source_mode="local"` と組み合わせると完全オフラインでも動作します。


In [ ]:
# 事前準備: ollama serve && ollama pull llama3.1:8b
config = create_config(
    provider="local",
    local_base_url="http://localhost:11434",
    local_backend="ollama",
    model="llama3.1:8b",
    source_mode="hybrid",           # ローカル文書 + Web
    output_format="markdown",
    output_dir="./output/local_llm",
)

result = DeepResearchTool(config).run(
    query="社内技術資料に基づく特許出願候補の洗い出し",
    additional_documents=["./docs/tech_memo.pdf"],
)
print("→", result["report_path"])